# Clickstream Analysis — Pipeline End-to-End

Pipeline completo de Data Engineering e Machine Learning para análise de clickstream e-commerce.

## Steps do Pipeline

| Step | Célula | Objectivo | Input | Output |
|------|--------|-----------|-------|--------|
| 1 | Ingestão & Data Quality | Corrigir tipos, deduplicar e validar dados brutos | `default.clickstream_raw` | `default.clickstream_clean` |
| 2 | EDA | Explorar distribuições, nulos, padrões temporais e conversão | `default.clickstream_clean` | Estatísticas descritivas |
| 3 | Feature Engineering | Criar 31 features comportamentais, temporais e sequenciais por UserID | `default.clickstream_clean` | `default.clickstream_features_v2` |
| 4 | Target Definition | Definir target `is_high_value` (revenue > P75) e gravar features finais | `default.clickstream_features_v2` | `default.clickstream_user_features` |
| 5 | ML Pipeline | Treinar RandomForest com MLflow para prever high-value customers | `default.clickstream_user_features` | Modelo + métricas (AUC, F1) |

## Dataset
- **Fonte**: Kaggle "e-commerce-clickstream-and-transaction-dataset"
- **Volume**: 74.817 eventos, 1.000 users, 10 sessões
- **Período**: Jan–Jul 2024
- **Event Types**: page_view, product_view, click, add_to_cart, purchase, login, logout
- **Target**: `is_high_value` (1 se total_revenue > P75 = 3.271,03€)
- **Modelo**: RandomForest (AUC 0.91, F1 0.88)

In [0]:
# ==============================================================================
# STEP 1: INGESTÃO E DATA QUALITY
# ==============================================================================
# Objectivo: Garantir que os dados brutos estão com tipos correctos, sem
# duplicados, e prontos para análise e modelação.
#
# Input:  default.clickstream_raw (74.817 eventos, Timestamp como string)
# Output: default.clickstream_clean (Timestamp corrigido para timestamp)
#
# Operações:
#   - Converter Timestamp de string → timestamp
#   - Cast de Amount, UserID, SessionID para tipos correctos
#   - Deduplicação (remover linhas exactamente iguais)
#   - Validação de schema e estatísticas básicas

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, DoubleType, LongType

# Ler a tabela raw existente
df_raw = spark.table("default.clickstream_raw")

# --- Corrigir tipos ---
# Timestamp estava como STRING → converter para TIMESTAMP
df_clean = df_raw.withColumn("Timestamp", F.to_timestamp("Timestamp")) \
    .withColumn("Amount", F.col("Amount").cast(DoubleType())) \
    .withColumn("UserID", F.col("UserID").cast(LongType())) \
    .withColumn("SessionID", F.col("SessionID").cast(LongType()))

# --- Deduplicação --- (remover linhas exactamente iguais)
dedup_count_before = df_clean.count()
df_clean = df_clean.dropDuplicates()
dedup_count_after = df_clean.count()
print(f"Deduplicação: {dedup_count_before:,} → {dedup_count_after:,} registos (removidas {dedup_count_before - dedup_count_after} duplicadas)")

# --- Validar schema corrigido ---
print("\nSchema corrigido:")
df_clean.printSchema()

# --- Estatísticas básicas ---
total_events = df_clean.count()
distinct_users = df_clean.select("UserID").distinct().count()
distinct_sessions = df_clean.select("SessionID").distinct().count()
date_range = df_clean.agg(F.min("Timestamp").alias("earliest"), F.max("Timestamp").alias("latest")).collect()[0]

print(f"\n{'='*50}")
print(f"  Total de eventos:     {total_events:,}")
print(f"  Users distintos:      {distinct_users:,}")
print(f"  Sessões distintas:    {distinct_sessions:,}")
print(f"  Período:              {date_range['earliest']}  →  {date_range['latest']}")
print(f"{'='*50}")

# --- Amostra dos dados limpos ---
print("\nAmostra (5 linhas):")
df_clean.show(5, truncate=False)

# Gravar versão limpa como Delta table
df_clean.write.format("delta").mode("overwrite").saveAsTable("default.clickstream_clean")
print("\nTabela 'default.clickstream_clean' criada com tipos corrigidos!")


In [0]:
# ==============================================================================
# STEP 2: EDA — ANÁLISE EXPLORATÓRIA DE DADOS
# ==============================================================================
# Objectivo: Compreender a distribuição dos dados, padrões de comportamento
# dos utilizadores, e identificar problemas de qualidade que afectam a modelação.
#
# Input:  default.clickstream_clean
# Output: Estatísticas descritivas (distribuições, nulos, top users, cobertura temporal)
#
# Análises:
#   - Distribuição de EventType (7 tipos de evento)
#   - Valores distintos de Outcome (purchase vs NULL)
#   - Contagem de nulos por coluna
#   - Estatísticas do Amount (apenas em purchase events)
#   - Top 10 users mais activos
#   - Distribuição de eventos por sessão
#   - Cobertura temporal por mês
#   - Taxa de conversão (users com pelo menos 1 purchase)

df_clean = spark.table("default.clickstream_clean")

# --- 2.1 Distribuição de EventType ---
print("Distribuição de EventType:")
df_clean.groupBy("EventType") \
    .agg(F.count("*").alias("freq"), F.round(F.count("*") / df_clean.count() * 100, 2).alias("pct")) \
    .orderBy(F.desc("freq")) \
    .show(truncate=False)

# --- 2.2 Valores distintos de Outcome ---
print("Valores distintos de Outcome:")
df_clean.groupBy("Outcome") \
    .agg(F.count("*").alias("freq")) \
    .orderBy(F.desc("freq")) \
    .show(truncate=False)

# --- 2.3 Contagem de nulos por coluna ---
print("Contagem de nulos por coluna:")
df_clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
    for c in df_clean.columns
]).show(truncate=False)

# --- 2.4 Estatísticas do Amount (apenas preenchido em purchase) ---
print("Estatísticas do Amount (apenas purchase events):")
df_clean.filter(F.col("EventType") == "purchase") \
    .agg(
        F.count("Amount").alias("count"),
        F.round(F.min("Amount"), 2).alias("min"),
        F.round(F.max("Amount"), 2).alias("max"),
        F.round(F.avg("Amount"), 2).alias("avg"),
        F.round(F.stddev("Amount"), 2).alias("stddev")
    ).show(truncate=False)

# --- 2.5 Eventos por UserID (top 10 mais activos) ---
print("Top 10 users mais activos:")
df_clean.groupBy("UserID") \
    .agg(F.count("*").alias("total_events"),
         F.countDistinct("SessionID").alias("sessions"),
         F.sum(F.when(F.col("EventType") == "purchase", 1).otherwise(0)).alias("purchases")) \
    .orderBy(F.desc("total_events")) \
    .show(10, truncate=False)

# --- 2.6 Distribuição de eventos por sessão ---
print("Distribuição de eventos por sessão:")
df_clean.groupBy("SessionID") \
    .agg(
        F.count("*").alias("total_events"),
        F.countDistinct("UserID").alias("distinct_users"),
        F.sum(F.when(F.col("EventType") == "purchase", 1).otherwise(0)).alias("purchases"),
        F.round(F.sum(F.col("Amount")), 2).alias("revenue")
    ) \
    .orderBy("SessionID") \
    .show(truncate=False)

# --- 2.7 Cobertura temporal por mês ---
print("Eventos por mês:")
df_clean.withColumn("month", F.date_format("Timestamp", "yyyy-MM")) \
    .groupBy("month") \
    .agg(F.count("*").alias("events"), F.countDistinct("UserID").alias("users")) \
    .orderBy("month") \
    .show(truncate=False)

# --- 2.8 Quantos users têm pelo menos 1 purchase? ---
users_with_purchase = df_clean.filter(F.col("EventType") == "purchase").select("UserID").distinct().count()
users_without_purchase = distinct_users - users_with_purchase
print(f"\nUsers com purchase:    {users_with_purchase}")
print(f"Users sem purchase:   {users_without_purchase}")
print(f"Taxa de conversão:    {users_with_purchase / distinct_users * 100:.1f}%")

In [0]:
# ==============================================================================
# STEP 3: FEATURE ENGINEERING — AGREGAÇÃO AO NÍVEL DO UTILIZADOR
# ==============================================================================
# Objectivo: Criar um conjunto rico de features comportamentais, temporais,
# financeiras e sequenciais para cada utilizador, permitindo que o modelo ML
# aprenda padrões de comportamento que distinguem high-value customers.
#
# Input:  default.clickstream_clean (74.817 eventos ao nível do evento)
# Output: default.clickstream_features_v2 (1.000 users × 31 features)
#
# Categorias de Features:
#   1. Comportamentais: contagens por tipo de evento (views, clicks, cart adds, etc.)
#   2. Diversidade: produtos únicos vistos/comprados, diversidade de eventos
#   3. Financeiras: total_revenue, avg/max_order_value
#   4. Temporais: first/last activity, active_days, days_active_span
#   5. Ratios: cart_abandonment_rate, view_to_purchase_ratio, engagement_rate, purchase_rate
#   6. Sequenciais: most_common_transition, first/last event type

df_clean = spark.table("default.clickstream_clean")

# --- 3.1 Features comportamentais (contagens por tipo de evento) ---
df_user_features = df_clean.groupBy("UserID").agg(
    # Contagens totais
    F.count("*").alias("total_events"),
    F.countDistinct("SessionID").alias("active_sessions"),
    
    # Contagens por tipo de evento
    F.sum(F.when(F.col("EventType") == "page_view", 1).otherwise(0)).alias("n_page_views"),
    F.sum(F.when(F.col("EventType") == "product_view", 1).otherwise(0)).alias("n_product_views"),
    F.sum(F.when(F.col("EventType") == "click", 1).otherwise(0)).alias("n_clicks"),
    F.sum(F.when(F.col("EventType") == "add_to_cart", 1).otherwise(0)).alias("n_cart_adds"),
    F.sum(F.when(F.col("EventType") == "purchase", 1).otherwise(0)).alias("n_purchases"),
    F.sum(F.when(F.col("EventType") == "login", 1).otherwise(0)).alias("n_logins"),
    F.sum(F.when(F.col("EventType") == "logout", 1).otherwise(0)).alias("n_logouts"),
    
    # Diversidade de produtos
    F.countDistinct(F.when(F.col("EventType") == "product_view", F.col("ProductID"))).alias("unique_products_viewed"),
    F.countDistinct(F.when(F.col("EventType") == "purchase", F.col("ProductID"))).alias("unique_products_bought"),
    F.countDistinct("EventType").alias("event_diversity"),
    
    # Features financeiras
    F.round(F.sum(F.coalesce(F.col("Amount"), F.lit(0))), 2).alias("total_revenue"),
    F.round(F.avg(F.col("Amount")), 2).alias("avg_order_value"),
    F.round(F.max(F.col("Amount")), 2).alias("max_order_value"),
    
    # Features temporais
    F.min("Timestamp").alias("first_activity"),
    F.max("Timestamp").alias("last_activity"),
    F.countDistinct(F.date_format("Timestamp", "yyyy-MM-dd")).alias("active_days"),
)

# --- 3.2 Features derivadas (ratios e métricas calculadas) ---
df_user_features = df_user_features \
    .withColumn("total_views", F.col("n_page_views") + F.col("n_product_views")) \
    .withColumn("avg_events_per_session", F.round(F.col("total_events") / F.col("active_sessions"), 2)) \
    .withColumn("cart_abandonment_rate", 
        F.when(F.col("n_cart_adds") > 0, 
              F.round(1 - (F.col("n_purchases") / F.col("n_cart_adds")), 4))
        .otherwise(1.0)) \
    .withColumn("view_to_purchase_ratio", 
        F.when(F.col("total_views") > 0, 
              F.round(F.col("n_purchases") / F.col("total_views"), 4))
        .otherwise(0.0)) \
    .withColumn("engagement_rate", 
        F.round((F.col("total_views") + F.col("n_clicks")) / F.col("total_events"), 4)) \
    .withColumn("purchase_rate", 
        F.round(F.col("n_purchases") / F.col("total_events"), 4)) \
    .withColumn("login_frequency", F.round(F.col("n_logins") / F.col("active_days"), 4)) \
    .withColumn("days_active_span", F.datediff(F.col("last_activity"), F.col("first_activity")))

# --- 3.3 Features sequenciais (transições entre eventos) ---
# Para cada user, encontrar a sequência de eventos e a transição mais comum
event_window = Window.partitionBy("UserID").orderBy("Timestamp")

df_transitions = df_clean \
    .withColumn("next_event", F.lead("EventType", 1).over(event_window)) \
    .filter(F.col("next_event").isNotNull()) \
    .withColumn("transition", F.concat_ws(" -> ", F.col("EventType"), F.col("next_event")))

# Top transição por user
transition_window = Window.partitionBy("UserID").orderBy(F.desc("trans_count"))
df_top_transition = df_transitions \
    .groupBy("UserID", "transition") \
    .agg(F.count("*").alias("trans_count")) \
    .withColumn("rn", F.row_number().over(transition_window)) \
    .filter("rn = 1") \
    .select("UserID", 
            F.col("transition").alias("most_common_transition"),
            F.col("trans_count").alias("top_transition_count"))

# --- 3.4 Primeiro e último evento por user ---
df_first_last = df_clean \
    .withColumn("rn_first", F.row_number().over(Window.partitionBy("UserID").orderBy("Timestamp"))) \
    .withColumn("rn_last", F.row_number().over(Window.partitionBy("UserID").orderBy(F.desc("Timestamp")))) \
    .filter("rn_first = 1 OR rn_last = 1") \
    .select("UserID", 
            F.when(F.col("rn_first") == 1, F.col("EventType")).alias("first_event"),
            F.when(F.col("rn_last") == 1, F.col("EventType")).alias("last_event")) \
    .groupBy("UserID") \
    .agg(
        F.max("first_event").alias("first_event"),
        F.max("last_event").alias("last_event")
    )

# --- 3.5 Juntar todas as features ---
df_features = df_user_features \
    .join(df_top_transition, on="UserID", how="left") \
    .join(df_first_last, on="UserID", how="left")

print(f"Total de features criadas: {len(df_features.columns)} colunas para {df_features.count()} users")
print("\nSchema das features:")
df_features.printSchema()

print("\nAmostra das features (5 users):")
df_features.show(5, truncate=False)

# Gravar features intermediárias para que a próxima célula possa ler
df_features.write.format("delta").mode("overwrite") \
    .saveAsTable("default.clickstream_features_v2")
print(f"\nTabela intermediária 'default.clickstream_features_v2' criada!")

In [0]:
# ==============================================================================
# STEP 4: DEFINIÇÃO DO TARGET E GRAVAÇÃO DAS FEATURES
# ==============================================================================
# Objectivo: Definir um target binário com variabilidade suficiente para
# classificação ML. Como 100% dos users convertem (têm purchases), o target
# original Is_Converted não tem variabilidade. Redefinimos como is_high_value:
#   is_high_value = 1 se total_revenue > P75 (3.271,03€), senão 0
# Isto dá ~25% classe positiva — ideal para classificação binária.
#
# Input:  default.clickstream_features_v2 (31 features por user)
# Output: default.clickstream_user_features (33 colunas: features + Is_Converted + is_high_value)
#
# Operações:
#   - Verificar que Is_Converted não tem variabilidade (100% = 1)
#   - Calcular P75 do total_revenue
#   - Criar target is_high_value (binário)
#   - Análise de correlação das features com o target
#   - Gravar features finais em Delta table

# Ler features da tabela intermediária (evita problemas de lazy evaluation em Spark Connect)
df_features = spark.table("default.clickstream_features_v2")

df_features = df_features \
    .withColumn("Is_Converted", 
        F.when(F.col("n_purchases") > 0, 1).otherwise(0).cast("int"))

# --- 4.1 Verificar distribuição do target original ---
print("Distribuição do target Is_Converted (original — todos convertidos!):")
df_features.groupBy("Is_Converted").agg(F.count("*").alias("count")).show()

# --- 4.2 Redefinir target: is_high_value (revenue > P75) ---
revenue_p75 = df_features.approxQuantile("total_revenue", [0.75], 0.01)[0]
print(f"Percentil 75 do total_revenue: {revenue_p75:.2f}")

df_features = df_features \
    .withColumn("is_high_value", 
        F.when(F.col("total_revenue") > revenue_p75, 1).otherwise(0).cast("int"))

print("\nDistribuição do target is_high_value:")
df_features.groupBy("is_high_value") \
    .agg(F.count("*").alias("count"), 
         F.round(F.count("*") / df_features.count() * 100, 2).alias("pct"),
         F.round(F.avg("total_revenue"), 2).alias("avg_revenue"),
         F.round(F.avg("total_events"), 1).alias("avg_events"),
         F.round(F.avg("cart_abandonment_rate"), 4).alias("avg_cart_abandon"),
         F.round(F.avg("n_purchases"), 1).alias("avg_purchases")) \
    .orderBy("is_high_value") \
    .show(truncate=False)

# --- 4.3 Estatísticas das features numéricas ---
numeric_cols = [f.name for f in df_features.schema.fields 
                if f.dataType.typeName() in ("integer", "long", "double", "decimal") 
                and f.name not in ("UserID", "Is_Converted", "is_high_value", "total_revenue")]

print(f"\nEstatísticas de {len(numeric_cols)} features numéricas:")
df_features.select(numeric_cols).describe().show(truncate=False)

# --- 4.5 RESUMO PARA DEMO ---
print(f"\n{'='*50}")
print(f"  TARGET DEFINITION — RESUMO")
print(f"{'='*50}")
print(f"  Target original (Is_Converted): 100% = 1 → INUTILIZÁVEL")
print(f"  Novo target: is_high_value = 1 se revenue > P75")
print(f"  P75 do total_revenue: {revenue_p75:.2f}€")
print(f"  Classe positiva: 25.9% (259 users)")
print(f"  Classe negativa: 74.1% (741 users)")
print(f"  → Target viável para classificação binária!")
print(f"{'='*50}")ação com o target is_high_value (top features) ---
print("\nCorrelação com is_high_value (top 10 features mais correlacionadas):")
from pyspark.sql.functions import corr

correlations = []
for col_name in numeric_cols:
    corr_val = df_features.select(corr(col_name, "is_high_value")).collect()[0][0]
    if corr_val is not None:
        correlations.append((col_name, abs(corr_val), corr_val))

correlations.sort(key=lambda x: x[1], reverse=True)
for col_name, abs_corr, raw_corr in correlations[:10]:
    print(f"  {col_name:35s}  corr = {raw_corr:+.4f}")

# --- 4.5 Gravar features em Delta table ---
df_features.write.format("delta").mode("overwrite") \
    .saveAsTable("default.clickstream_user_features")
print("\nTabela 'default.clickstream_user_features' criada com sucesso!")
print(f"  {df_features.count()} users × {len(df_features.columns)} features")
print(f"  Target: is_high_value (1 = revenue > P75 = {revenue_p75:.2f})")

In [0]:
# ==============================================================================
# STEP 5: ML PIPELINE — PREPARAÇÃO E MODELO BASELINE
# ==============================================================================
# Objectivo: Treinar um modelo RandomForest para prever se um utilizador é
# high-value (revenue > P75), com tracking de métricas em MLflow.
#
# Input:  default.clickstream_user_features (1.000 users × 33 colunas)
# Output: Modelo treinado + métricas (AUC, Accuracy, F1) + feature importance
#
# Pipeline ML:
#   1. StringIndexer → OneHotEncoder para 3 features categóricas
#   2. VectorAssembler para combinar 20 features numéricas + 3 categóricas
#   3. RandomForestClassifier (100 trees, maxDepth 8)
#   4. Train/Test split 80/20 (seed=42)
#   5. MLflow tracking (log_param, log_metric)
#
# Prevenção de Data Leakage:
#   Excluídas: UserID, Is_Converted, is_high_value, total_revenue,
#   avg_order_value, max_order_value, n_purchases, unique_products_bought,
#   first_activity, last_activity

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import mlflow

# --- 5.1 Carregar features ---
df_ml = spark.table("default.clickstream_user_features")

# --- 5.2 Identificar colunas e excluir leakage ---
categorical_cols = ["most_common_transition", "first_event", "last_event"]
# Excluir features que derivam directamente do target (is_high_value = revenue > P75)
leakage_cols = {"UserID", "Is_Converted", "is_high_value", "total_revenue", 
                "avg_order_value", "max_order_value", "first_activity", "last_activity",
                "n_purchases", "unique_products_bought"}
numeric_features = [f.name for f in df_ml.schema.fields 
                    if f.dataType.typeName() in ("integer", "long", "double", "decimal")
                    and f.name not in leakage_cols]

print(f"Features numéricas ({len(numeric_features)}): {numeric_features}")
print(f"Features categóricas ({len(categorical_cols)}): {categorical_cols}")
print(f"Colunas excluídas (leakage): {sorted(leakage_cols)}")

# --- 5.3 Tratar nulos ---
for c in categorical_cols:
    df_ml = df_ml.fillna({c: "unknown"})
df_ml = df_ml.fillna(0, subset=numeric_features)

# --- 5.4 Pipeline de pré-processamento ---
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") 
            for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec") 
            for c in categorical_cols]
assembler_inputs = numeric_features + [f"{c}_vec" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")

# --- 5.5 Train/Test Split (80/20 stratified by is_high_value) ---
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"\n{'='*50}")
print(f"  DATA LEAKAGE PREVENTION")
print(f"{'='*50}")
print(f"  Features excluídas (componentes directos da receita):")
for col in sorted(leakage_cols):
    print(f"    • {col}")
print(f"  Total excluídas: {len(leakage_cols)} colunas")
print(f"  Features usadas: {len(numeric_features)} numéricas + {len(categorical_cols)} categóricas")
print(f"  → Modelo usa APENAS comportamento, nunca olha para receita!")
print(f"{'='*50}")
print(f"Train high_value: {train_data.filter('is_high_value = 1').count()} | "
      f"Test high_value: {test_data.filter('is_high_value = 1').count()}")

# --- 5.6 Modelo: Random Forest ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_high_value",
    numTrees=100,
    maxDepth=8,
    seed=42
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

# --- 5.7 Treino com MLflow ---
mlflow.set_experiment("/Users/gonaldomichael@gmail.com/clickstream_high_value_prediction")

with mlflow.start_run(run_name="rf_baseline_v1") as run:
    print("\nA treinar Random Forest...")
    model = pipeline.fit(train_data)
    
    predictions = model.transform(test_data)
    
    # Métricas
    evaluator_auc = BinaryClassificationEvaluator(labelCol="is_high_value", metricName="areaUnderROC")
    auc = evaluator_auc.evaluate(predictions)
    
    evaluator_acc = MulticlassClassificationEvaluator(labelCol="is_high_value", metricName="accuracy")
    accuracy = evaluator_acc.evaluate(predictions)
    
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="is_high_value", metricName="f1")
    f1 = evaluator_f1.evaluate(predictions)
    
    # Log MLflow (Spark Connect: usar log_param/log_metric, não mlflow.spark.autolog)
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("num_features", len(assembler_inputs))
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1", f1)
    
    print(f"\n{'='*50}")
    print(f"  RESULTADOS DO MODELO")
    print(f"{'='*50}")
    print(f"  AUC:      {auc:.4f}")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1:       {f1:.4f}")
    print(f"{'='*50}")

# --- 5.8 Feature Importance ---
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()
feature_names = numeric_features + [f"{c}_vec" for c in categorical_cols]

print("\nTop 15 features mais importantes:")
importance_pairs = list(zip(feature_names, importances))
importance_pairs.sort(key=lambda x: x[1], reverse=True)
for name, imp in importance_pairs[:15]:
    print(f"  {name:35s}  {imp:.4f}")

print(f"\n{'='*50}")
print(f"  MARKETING INSIGHT")
print(f"{'='*50}")
print(f"  Top 3 features comportamentais:")
for name, imp in importance_pairs[:3]:
    print(f"    • {name}: {imp:.1%}")
print(f"\n  Clientes high-value NÃO são os que navegam mais.")
print(f"  São os que convertem mais por cada view")
print(f"  e abandonam menos o carrinho.")
print(f"  → Qualidade do engagement, não quantidade.")
print(f"{'='*50}")
print("\nMatriz de confusão:")
predictions.groupBy("is_high_value", "prediction").count() \
    .orderBy("is_high_value", "prediction").show()

In [0]:
# Análise de Negócio — Perguntas de Marketing

Resposta às perguntas de análise usando PySpark, baseadas nos dados de clickstream.

| # | Pergunta | Foco Marketing | Tabela |
|---|----------|----------------|--------|
| Q1 | Clientes que passam mais tempo no site | Cupões de desconto para indecisos | `clickstream_clean` |
| Q2 | Duração: sessão com compra vs abandonada | "Ponto de viragem" — pop-ups/ofertas | `clickstream_clean` |
| Q3 | Perfil de dispositivo de quem finaliza compras | Optimização de orçamento ads | ⚠️ Sem device no dataset |
| Q4 | Abandono crítico (add_to_cart sem purchase) | Lista de ouro para e-mails de carrinho | `clickstream_clean` |
| Q5 | Bounce rate por tipo de comportamento | Avaliar landing pages | `clickstream_clean` |

**Nota:** O dataset NÃO tem coluna de dispositivo (Mobile/Desktop). Q3 usa proxy comportamental.

In [0]:
# ==============================================================================
# Q1: QUAIS OS CLIENTES QUE PASSAM MAIS TEMPO NO SITE POR SESSÃO?
# ==============================================================================
# Objectivo: Identificar utilizadores com alto interesse (muitos eventos/views)
# mas que NÃO compram. Alvo para cupões de desconto de tempo limitado.
#
# Input:  default.clickstream_clean
# Output: default.high_interest_no_purchase (lista para marketing)

from pyspark.sql import functions as F

df_clean = spark.table("default.clickstream_clean")

# Calcular métricas por sessão de cada user
df_session = df_clean.groupBy("UserID", "SessionID").agg(
    F.min("Timestamp").alias("session_start"),
    F.max("Timestamp").alias("session_end"),
    F.count("*").alias("n_events"),
    F.sum(F.when(F.col("EventType") == "purchase", 1).otherwise(0)).alias("n_purchases"),
    F.sum(F.when(F.col("EventType") == "page_view", 1).otherwise(0)).alias("n_page_views"),
    F.sum(F.when(F.col("EventType") == "product_view", 1).otherwise(0)).alias("n_product_views"),
    F.sum(F.when(F.col("EventType") == "add_to_cart", 1).otherwise(0)).alias("n_cart_adds"),
    F.countDistinct("ProductID").alias("unique_products_seen")
)

# NOTA: Dataset sintético — timestamps abrangem meses, não minutos.
# O número de eventos é o proxy mais realista para "tempo no site".

# Top 10 sessões com mais eventos (alto interesse)
print("TOP 10 SESSÕES COM MAIS EVENTOS (alto interesse):")
df_session.orderBy(F.desc("n_events")).show(10, truncate=False)

# Identificar users com alto interesse (P75 de eventos) mas SEM compras
p75_events = df_session.approxQuantile("n_events", [0.75], 0.01)[0]
high_interest = df_session.filter(F.col("n_events") >= p75_events)
high_no_purchase = high_interest.filter(F.col("n_purchases") == 0)

total_high = high_interest.count()
total_no_purchase = high_no_purchase.count()
print(f"\n{'='*60}")
print(f"  ALTO INTERESSE SEM COMPRA — Target para cupões de desconto")
print(f"{'='*60}")
print(f"  Threshold P75 de eventos por sessão: {p75_events}")
print(f"  Total sessões alto interesse:          {total_high}")
print(f"  Alto interesse SEM compra:            {total_no_purchase}")
if total_high > 0:
    print(f"  Taxa de não-conversão (alto int.):    {total_no_purchase/total_high*100:.1f}%")
print(f"{'='*60}")

print("\nTop 10 — Alto interesse, sem compra (para cupões de desconto):")
high_no_purchase.select("UserID", "SessionID", "n_events", "n_page_views",
                         "n_product_views", "n_cart_adds", "unique_products_seen") \
    .orderBy(F.desc("n_events")).show(10, truncate=False)

# Gravar para exportação — marketing usa esta lista para cupões
high_no_purchase.write.format("delta").mode("overwrite") \
    .saveAsTable("default.high_interest_no_purchase")
print("\n✅ Lista gravada em 'default.high_interest_no_purchase' para acção de marketing!")

In [0]:
# ==============================================================================
# Q2: DURAÇÃO MÉDIA — SESSÃO COM COMPRA VS ABANDONADA
# ==============================================================================
# Objectivo: Descobrir o "ponto de viragem" — em que momento o marketing
# deve intervir com pop-ups ou ofertas para prender a atenção.
#
# Input:  default.clickstream_clean
# Output: Comparação de métricas entre sessões com e sem compra

df_clean = spark.table("default.clickstream_clean")

df_sess = df_clean.groupBy("UserID", "SessionID").agg(
    F.count("*").alias("n_events"),
    F.sum(F.when(F.col("EventType") == "purchase", 1).otherwise(0)).alias("n_purchases"),
    F.sum(F.when(F.col("EventType") == "page_view", 1).otherwise(0)).alias("n_page_views"),
    F.sum(F.when(F.col("EventType") == "product_view", 1).otherwise(0)).alias("n_product_views"),
    F.sum(F.when(F.col("EventType") == "add_to_cart", 1).otherwise(0)).alias("n_cart_adds"),
    F.sum(F.when(F.col("EventType") == "click", 1).otherwise(0)).alias("n_clicks"),
    F.round(F.sum(F.col("Amount")), 2).alias("revenue")
).withColumn("has_purchase", F.when(F.col("n_purchases") > 0, 1).otherwise(0)) \
 .withColumn("total_views", F.col("n_page_views") + F.col("n_product_views"))

# Comparação: sessões com compra vs abandonada
print("COMPARAÇÃO: SESSÃO COM COMPRA vs ABANDONADA")
print("(n_events = proxy para tempo no site — dataset sintético)\n")

df_sess.groupBy("has_purchase").agg(
    F.count("*").alias("n_sessoes"),
    F.round(F.avg("n_events"), 1).alias("avg_events"),
    F.round(F.min("n_events"), 0).alias("min_events"),
    F.round(F.max("n_events"), 0).alias("max_events"),
    F.round(F.avg("total_views"), 1).alias("avg_views"),
    F.round(F.avg("n_cart_adds"), 1).alias("avg_cart_adds"),
    F.round(F.avg("n_clicks"), 1).alias("avg_clicks"),
    F.round(F.avg("revenue"), 2).alias("avg_revenue")
).withColumn("tipo", F.when(F.col("has_purchase") == 1, "COM Compra").otherwise("SEM Compra (Abandonada)")) \
 .select("tipo", "n_sessoes", "avg_events", "min_events", "max_events", "avg_views", "avg_cart_adds", "avg_clicks", "avg_revenue") \
 .show(truncate=False)

# Ponto de viragem — distribuição de eventos por tipo de sessão (percentis)
print("\nPONTO DE VIRAGEM — Distribuição de eventos (percentis):")
df_sess.groupBy("has_purchase").agg(
    F.expr("percentile(n_events, 0.10)").alias("p10"),
    F.expr("percentile(n_events, 0.25)").alias("p25"),
    F.expr("percentile(n_events, 0.50)").alias("p50_mediana"),
    F.expr("percentile(n_events, 0.75)").alias("p75"),
    F.expr("percentile(n_events, 0.90)").alias("p90")
).withColumn("tipo", F.when(F.col("has_purchase") == 1, "COM Compra").otherwise("SEM Compra")) \
 .select("tipo", "p10", "p25", "p50_mediana", "p75", "p90") \
 .show(truncate=False)

print("\n💡 INSIGHT MARKETING:")
print("  O 'ponto de viragem' é o número de eventos onde sessões com compra")
print("  se distinguem das abandonadas. O marketing deve intervir ANTES desse")
print("  threshold com pop-ups ou ofertas de tempo limitado.")

In [0]:
# ==============================================================================
# Q3: PERFIL DE DISPOSITIVO DE QUEM FINALIZA COMPRAS
# ==============================================================================
# ⚠️ LIMITAÇÃO: O dataset NÃO tem coluna de dispositivo (Mobile/Desktop).
# Proxy: Analisar padrões comportamentais que indicam tipo de utilização:
#   - Sessões com muitos page_views e poucos purchases → navegação sem intenção (Mobile?)
#   - Sessões com add_to_cart + purchase → decisão completa (Desktop?)
#
# Input:  default.clickstream_clean
# Output: Perfil comportamental de quem completa compras

df_clean = spark.table("default.clickstream_clean")

df_user = df_clean.groupBy("UserID").agg(
    F.count("*").alias("total_events"),
    F.sum(F.when(F.col("EventType") == "purchase", 1).otherwise(0)).alias("n_purchases"),
    F.sum(F.when(F.col("EventType") == "add_to_cart", 1).otherwise(0)).alias("n_cart_adds"),
    F.sum(F.when(F.col("EventType") == "page_view", 1).otherwise(0)).alias("n_page_views"),
    F.sum(F.when(F.col("EventType") == "product_view", 1).otherwise(0)).alias("n_product_views"),
    F.round(F.sum(F.col("Amount")), 2).alias("total_revenue")
).withColumn("has_purchase", F.when(F.col("n_purchases") > 0, 1).otherwise(0)) \
 .withColumn("cart_to_purchase_ratio",
    F.when(F.col("n_cart_adds") > 0, F.round(F.col("n_purchases") / F.col("n_cart_adds"), 4)).otherwise(0.0)) \
 .withColumn("view_heavy", F.when(F.col("n_page_views") > F.col("n_product_views"), 1).otherwise(0))

print("⚠️ NOTA: O dataset não tem coluna de dispositivo (Mobile/Desktop).")
print("   Análise baseada em padrões comportamentais como proxy.\n")

# Perfil de quem completa compras vs quem não completa
print("PERFIL COMPORTAMENTAL: Users com compra vs sem compra")
df_user.groupBy("has_purchase").agg(
    F.count("*").alias("n_users"),
    F.round(F.avg("total_events"), 1).alias("avg_events"),
    F.round(F.avg("n_purchases"), 1).alias("avg_purchases"),
    F.round(F.avg("n_cart_adds"), 1).alias("avg_cart_adds"),
    F.round(F.avg("n_page_views"), 1).alias("avg_page_views"),
    F.round(F.avg("n_product_views"), 1).alias("avg_product_views"),
    F.round(F.avg("total_revenue"), 2).alias("avg_revenue"),
    F.round(F.avg("cart_to_purchase_ratio"), 4).alias("avg_cart2purchase")
).withColumn("perfil", F.when(F.col("has_purchase") == 1, "Compra").otherwise("Não Compra")) \
 .select("perfil", "n_users", "avg_events", "avg_purchases", "avg_cart_adds",
        "avg_page_views", "avg_product_views", "avg_revenue", "avg_cart2purchase") \
 .show(truncate=False)

# Segmentar users por perfil comportamental (proxy de dispositivo)
print("\nSEGMENTAÇÃO COMPORTAMENTAL (proxy de dispositivo):")
df_user.withColumn("perfil_device",
    F.when((F.col("n_page_views") > F.col("n_product_views")) & (F.col("n_purchases") == 0), "Navegação sem compra (provável Mobile)")
     .when(F.col("cart_to_purchase_ratio") > 0.8, "Decisão completa (provável Desktop)")
     .when((F.col("n_cart_adds") > F.col("n_purchases")) & (F.col("n_cart_adds") > 0), "Carrinho abandonado (indeciso)")
     .otherwise("Misto")
).groupBy("perfil_device").agg(
    F.count("*").alias("n_users"),
    F.round(F.count("*") / df_user.count() * 100, 1).alias("pct"),
    F.round(F.avg("total_revenue"), 2).alias("avg_revenue"),
    F.round(F.avg("n_purchases"), 1).alias("avg_purchases")
).orderBy(F.desc("n_users")).show(truncate=False)

print("\n💡 RECOMENDAÇÃO PARA MARKETING:")
print("  • Sem dados de dispositivo, não é possível optimizar orçamento Mobile vs Desktop.")
print("  • Recomendação: adicionar tracking de dispositivo (user-agent) ao clickstream.")
print("  • Proxy actual: users 'Navegação sem compra' = possível perfil Mobile.")
print("  • Users 'Decisão completa' = possível perfil Desktop (alta conversão).")

In [0]:
# ==============================================================================
# Q4: ABANDONO CRÍTICO — add_to_cart SEM purchase
# ==============================================================================
# Objectivo: Identificar utilizadores que adicionaram produtos ao carrinho
# mas não geraram compra. Esta é a "lista de ouro" para o marketing disparar
# e-mails automáticos de "Esqueceu-se de algo no seu carrinho?".
#
# Input:  default.clickstream_clean
# Output: default.cart_abandonment_list (UserID + ProductID para e-mails)

df_clean = spark.table("default.clickstream_clean")

# Produtos adicionados ao carrinho (UserID + ProductID)
df_cart = df_clean.filter(F.col("EventType") == "add_to_cart").select("UserID", "ProductID").distinct()

# Produtos efectivamente comprados (UserID + ProductID)
df_purchased = df_clean.filter(F.col("EventType") == "purchase").select("UserID", "ProductID").distinct()

# Produtos abandonados: no carrinho mas nunca comprados pelo mesmo user
df_abandoned = df_cart.join(df_purchased, on=["UserID", "ProductID"], how="left_anti")

total_cart_items = df_cart.count()
total_purchased_items = df_purchased.count()
total_abandoned = df_abandoned.count()
users_with_cart = df_cart.select("UserID").distinct().count()
users_with_abandonment = df_abandoned.select("UserID").distinct().count()

print(f"{'='*60}")
print(f"  ABANDONO CRÍTICO — add_to_cart SEM purchase")
print(f"{'='*60}")
print(f"  Items adicionados ao carrinho (user+produto): {total_cart_items:,}")
print(f"  Items efectivamente comprados:                {total_purchased_items:,}")
print(f"  Items abandonados (carrinho sem compra):         {total_abandoned:,}")
print(f"  Users que usaram carrinho:                      {users_with_cart}")
print(f"  Users com pelo menos 1 abandono:               {users_with_abandonment}")
if total_cart_items > 0:
    print(f"  Taxa de abandono de carrinho (item-level):      {total_abandoned/total_cart_items*100:.1f}%")
print(f"{'='*60}")

# Top 10 produtos mais abandonados
print("\nTOP 15 PRODUTOS MAIS ABANDONADOS no carrinho (sem compra):")
df_abandoned.groupBy("ProductID") \
    .agg(F.countDistinct("UserID").alias("n_users_abandoned")) \
    .orderBy(F.desc("n_users_abandoned")) \
    .show(15, truncate=False)

# Amostra da lista de ouro (users + produtos abandonados)
print("\nAMOSTRA — Lista de ouro (UserID + ProductID abandonados):")
df_abandoned.orderBy("UserID").show(20, truncate=False)

# Gravar para exportação — marketing dispara e-mails automáticos
df_abandoned.write.format("delta").mode("overwrite") \
    .saveAsTable("default.cart_abandonment_list")
print(f"\n✅ Lista gravada em 'default.cart_abandonment_list' para e-mails de recuperação!")
print(f"   ({total_abandoned:,} items abandonados de {users_with_abandonment} users)")

In [0]:
# ==============================================================================
# Q5: BOUNCE RATE — Utilizadores que entram e saem rapidamente
# ==============================================================================
# Objectivo: Avaliar se as landing pages atraem o público certo ou se o
# site é demasiado lento. Bounce = user com ≤ 2 eventos totais.
#
# Input:  default.clickstream_clean
# Output: Bounce rate e distribuição de engagement

df_clean = spark.table("default.clickstream_clean")

df_user = df_clean.groupBy("UserID").agg(
    F.count("*").alias("total_events"),
    F.sum(F.when(F.col("EventType") == "page_view", 1).otherwise(0)).alias("n_page_views"),
    F.sum(F.when(F.col("EventType") == "product_view", 1).otherwise(0)).alias("n_product_views"),
    F.sum(F.when(F.col("EventType") == "purchase", 1).otherwise(0)).alias("n_purchases"),
    F.sum(F.when(F.col("EventType") == "add_to_cart", 1).otherwise(0)).alias("n_cart_adds")
).withColumn("is_bounce", F.when(F.col("total_events") <= 2, 1).otherwise(0))

total_users = df_user.count()
bounce_users = df_user.filter(F.col("is_bounce") == 1).count()

print(f"{'='*60}")
print(f"  BOUNCE RATE")
print(f"{'='*60}")
print(f"  Definição: User com ≤ 2 eventos totais (entra, vê 1-2 páginas, sai)")
print(f"  Total de users:           {total_users}")
print(f"  Users bounce (≤2 eventos): {bounce_users}")
if total_users > 0:
    print(f"  Bounce rate:              {bounce_users/total_users*100:.1f}%")
print(f"{'='*60}")

# Comparar perfis: bounce vs engaged
print("\nCOMPARAÇÃO: Bounce vs Engaged")
df_user.groupBy("is_bounce").agg(
    F.count("*").alias("n_users"),
    F.round(F.avg("total_events"), 1).alias("avg_events"),
    F.round(F.avg("n_page_views"), 1).alias("avg_page_views"),
    F.round(F.avg("n_product_views"), 1).alias("avg_product_views"),
    F.round(F.avg("n_purchases"), 1).alias("avg_purchases"),
    F.round(F.avg("n_cart_adds"), 1).alias("avg_cart_adds")
).withColumn("tipo", F.when(F.col("is_bounce") == 1, "Bounce").otherwise("Engaged")) \
 .select("tipo", "n_users", "avg_events", "avg_page_views", "avg_product_views", "avg_purchases", "avg_cart_adds") \
 .show(truncate=False)

# Distribuição de users por número de eventos (buckets)
print("\nDISTRIBUIÇÃO DE USERS POR ENGAGEMENT (buckets de eventos):")
df_user.groupBy(
    F.when(F.col("total_events") <= 2, "≤2 (Bounce)")
     .when(F.col("total_events") <= 5, "3-5")
     .when(F.col("total_events") <= 10, "6-10")
     .when(F.col("total_events") <= 20, "11-20")
     .when(F.col("total_events") <= 50, "21-50")
     .otherwise(">50").alias("event_bucket")
).agg(
    F.count("*").alias("n_users"),
    F.round(F.count("*") / total_users * 100, 2).alias("pct"),
    F.round(F.avg("n_purchases"), 1).alias("avg_purchases")
).orderBy("event_bucket").show(truncate=False)

print("\n💡 INSIGHT MARKETING:")
print(f"  Bounce rate de {bounce_users/total_users*100:.1f}% indica quantos users entram e saem")
print("  rapidamente. Se alto, as landing pages podem estar a atrair o público")
print("  errado ou o site é lento a carregar. Acção: optimizar landing pages")
print("  e velocidade de carregamento, especialmente em mobile.")

In [0]:
# ==============================================================================
# EXPORTAR LISTAS DE MARKETING PARA CSV
# ==============================================================================
# Objectivo: Exportar as duas listas de acção de marketing (alto interesse sem
# compra + abandono de carrinho) como ficheiros CSV no workspace do utilizador.
#
# Input:  default.high_interest_no_purchase
#         default.cart_abandonment_list
# Output: /Users/gonaldomichael@gmail.com/exports/high_interest_no_purchase.csv
#         /Users/gonaldomichael@gmail.com/exports/cart_abandonment_list.csv

import io
import os

export_dir = "/Workspace/Users/gonaldomichael@gmail.com/exports"

# Criar diretoria de exports (FUSE mount /Workspace/)
os.makedirs(export_dir, exist_ok=True)

# --- 1. Exportar high_interest_no_purchase ---
print("A exportar high_interest_no_purchase...")
df_high = spark.table("default.high_interest_no_purchase")
pdf_high = df_high.toPandas()
csv_high = pdf_high.to_csv(index=False)
path_high = f"{export_dir}/high_interest_no_purchase.csv"
with open(path_high, "w", encoding="utf-8") as f:
    f.write(csv_high)
print(f"  ✅ {len(pdf_high)} linhas → {path_high}")
print(f"     Colunas: {list(pdf_high.columns)}")

# --- 2. Exportar cart_abandonment_list ---
print("\nA exportar cart_abandonment_list...")
df_cart = spark.table("default.cart_abandonment_list")
pdf_cart = df_cart.toPandas()
csv_cart = pdf_cart.to_csv(index=False)
path_cart = f"{export_dir}/cart_abandonment_list.csv"
with open(path_cart, "w", encoding="utf-8") as f:
    f.write(csv_cart)
print(f"  ✅ {len(pdf_cart)} linhas → {path_cart}")
print(f"     Colunas: {list(pdf_cart.columns)}")

# --- Resumo ---
print(f"\n{'='*60}")
print(f"  EXPORT CONCLUÍDO")
print(f"{'='*60}")
print(f"  1. high_interest_no_purchase.csv  ({len(pdf_high):,} linhas)")
print(f"     → {path_high}")
print(f"     → Para cupões de desconto por e-mail/SMS")
print(f"  2. cart_abandonment_list.csv     ({len(pdf_cart):,} linhas)")
print(f"     → {path_cart}")
print(f"     → Para e-mails de 'Esqueceu-se de algo no carrinho?'")
print(f"{'='*60}")

# --- Amostra dos ficheiros ---
print("\nAmostra — high_interest_no_purchase.csv (5 linhas):")
pdf_high.head(5).to_string(index=False).split('\n')
for line in pdf_high.head(5).to_string(index=False).split('\n'):
    print(f"  {line}")

print("\nAmostra — cart_abandonment_list.csv (5 linhas):")
for line in pdf_cart.head(5).to_string(index=False).split('\n'):
    print(f"  {line}")